# 🛡️ PhishGuard — Intelligent Phishing Detection System
**Author:** Tushar Solanki  
**Project Type:** Educational Cybersecurity Prototype  
**Technology:** Python · Jupyter Notebook · SQLite · ipywidgets  

---

> ⚠️ **Disclaimer:** PhishGuard is a heuristic-based educational tool. It **cannot guarantee** that any URL is safe or malicious.  
> Heuristic detection can produce **false positives** (flagging safe URLs as risky) and **false negatives** (missing real phishing URLs).  
> Always verify suspicious URLs through official cybersecurity channels before acting on them.  
> This tool **never visits or opens** a scanned URL automatically.

## 📖 1. Introduction

Phishing is one of the most prevalent cybersecurity threats worldwide. Attackers craft malicious URLs that mimic legitimate websites to steal credentials, financial information, and personal data. According to the Anti-Phishing Working Group (APWG), hundreds of thousands of phishing attacks are reported every month.

**PhishGuard** is an intelligent URL analysis system built as an educational prototype to demonstrate how phishing URLs can be detected through:
- Structural and lexical analysis of URL components
- Heuristic risk scoring based on known phishing patterns
- Optional integration with threat-intelligence APIs
- Interactive visualization and scan-history management

---

## 🎯 2. Problem Statement

Traditional blacklist-based approaches to phishing detection fail against newly created phishing domains. A user who receives an unknown URL in an email or chat message has no easy way to assess its risk without visiting it — which itself is dangerous. PhishGuard addresses this gap by performing **offline structural analysis** before any network request is made.

---

## ✅ 3. Objectives

1. Extract security-relevant features from raw URLs without visiting them.
2. Compute a transparent, explainable risk score.
3. Classify URLs as **Low Risk**, **Suspicious**, or **High Risk**.
4. Optionally enrich results with threat-intelligence API data.
5. Persist scan history in a local SQLite database.
6. Provide interactive dashboards and exportable reports.
7. Include automated tests covering core logic.

---

## 🏗️ 4. Architecture Overview

```
┌─────────────────────────────────────────────────────────┐
│                     PhishGuard System                   │
│                                                         │
│  ┌──────────┐   ┌──────────────┐   ┌────────────────┐  │
│  │  URL     │──▶│  Feature     │──▶│  Risk          │  │
│  │  Input   │   │  Extractor   │   │  Classifier    │  │
│  └──────────┘   └──────────────┘   └────────┬───────┘  │
│                                             │           │
│  ┌──────────┐   ┌──────────────┐   ┌────────▼───────┐  │
│  │  Export  │◀──│  Dashboard   │◀──│  Scan Result   │  │
│  │ CSV/JSON │   │  + Charts    │   │  + Reasons     │  │
│  └──────────┘   └──────────────┘   └────────┬───────┘  │
│                                             │           │
│  ┌──────────┐   ┌──────────────┐            │           │
│  │  SQLite  │◀──│  History     │◀───────────┘           │
│  │   DB     │   │  Manager     │                        │
│  └──────────┘   └──────────────┘                        │
│                                                         │
│  ┌───────────────────────────────────────────────────┐  │
│  │  Optional: Threat Intelligence API (VirusTotal)   │  │
│  └───────────────────────────────────────────────────┘  │
└─────────────────────────────────────────────────────────┘
```

---

## 🔧 5. Technologies Used

| Library | Purpose |
|---|---|
| `urllib.parse` | URL parsing |
| `re` | Regex pattern matching |
| `sqlite3` | Local scan history database |
| `ipywidgets` | Interactive UI widgets |
| `matplotlib` | Charts and visualizations |
| `requests` | Optional HTTP API calls |
| `json`, `csv` | Export functionality |
| `unittest` | Automated testing |

---

## ⚠️ 6. Limitations

- **Heuristic scoring** is not ML-based and will produce false positives and false negatives.
- **No URL is visited** during analysis, so dynamic page content is never inspected.
- **Threat intelligence** requires a valid API key and internet access; results vary by provider.
- Short or newly registered domains may score low risk even if malicious (zero-day phishing).
- The tool is designed for **educational use only** and should not replace professional security tools.

In [ ]:
# ============================================================
#  CELL 1 — Imports & Configuration
# ============================================================
import re
import json
import csv
import sqlite3
import hashlib
import datetime
import unittest
import io
import sys
import os
import time
import warnings
from urllib.parse import urlparse, urlunparse, urlencode
from dataclasses import dataclass, field, asdict
from typing import List, Dict, Optional, Tuple
from pathlib import Path

# Optional imports — handled gracefully if missing
try:
    import requests
    REQUESTS_AVAILABLE = True
except ImportError:
    REQUESTS_AVAILABLE = False
    print("[INFO] 'requests' not installed. Threat-intelligence API calls will be disabled.")

try:
    import matplotlib
    import matplotlib.pyplot as plt
    import matplotlib.patches as mpatches
    matplotlib.use('Agg')          # non-interactive backend works in all environments
    MATPLOTLIB_AVAILABLE = True
except ImportError:
    MATPLOTLIB_AVAILABLE = False
    print("[INFO] 'matplotlib' not installed. Charts will be disabled.")

try:
    import ipywidgets as widgets
    from IPython.display import display, HTML, clear_output
    WIDGETS_AVAILABLE = True
except ImportError:
    WIDGETS_AVAILABLE = False
    print("[INFO] 'ipywidgets' not installed. Interactive widgets will be disabled.")

# ----------------------------------------------------------
#  Global configuration
# ----------------------------------------------------------
DB_PATH = "phishguard_history.db"

# Provide your VirusTotal API key here (optional).
# Leave empty string to skip threat-intelligence lookups.
VIRUSTOTAL_API_KEY = ""

# Thresholds for risk classification
RISK_THRESHOLDS = {
    "low":        (0,  39),   # score 0–39  → Low Risk
    "suspicious": (40, 69),   # score 40–69 → Suspicious
    "high":       (70, 100),  # score 70+   → High Risk
}

# Suspicious keywords often found in phishing URLs
SUSPICIOUS_KEYWORDS = [
    "login", "signin", "verify", "secure", "account", "update",
    "confirm", "banking", "password", "credential", "paypal",
    "apple", "microsoft", "google", "amazon", "facebook",
    "ebay", "netflix", "support", "helpdesk", "webscr",
    "wallet", "crypto", "recover", "suspended", "validate",
    "authenticate", "urgent", "alert", "limited", "winner",
]

# Common benign TLDs; unusual ones add risk
COMMON_TLDS = {
    ".com", ".org", ".net", ".edu", ".gov",
    ".co.uk", ".co.in", ".io", ".dev",
}

print("✅ PhishGuard imports and configuration loaded.")
print(f"   requests    : {'available' if REQUESTS_AVAILABLE else 'not installed'}")
print(f"   matplotlib  : {'available' if MATPLOTLIB_AVAILABLE else 'not installed'}")
print(f"   ipywidgets  : {'available' if WIDGETS_AVAILABLE else 'not installed'}")

In [ ]:
# ============================================================
#  CELL 2 — URL Feature Extraction Engine
# ============================================================

@dataclass
class URLFeatures:
    """All extracted features for a single URL."""
    raw_url:             str  = ""
    scheme:              str  = ""
    hostname:            str  = ""
    path:                str  = ""
    query:               str  = ""
    fragment:            str  = ""
    url_length:          int  = 0
    is_https:            bool = False
    is_ip_hostname:      bool = False
    has_at_symbol:       bool = False
    has_double_slash:    bool = False
    subdomain_count:     int  = 0
    suspicious_kw_count: int  = 0
    suspicious_kw_found: List[str] = field(default_factory=list)
    has_hex_encoding:    bool = False
    has_port:            bool = False
    port:                Optional[int] = None
    has_tld_in_path:     bool = False
    unusual_tld:         bool = False
    tld:                 str  = ""
    dash_count_hostname: int  = 0
    dot_count_hostname:  int  = 0
    digit_count_hostname:int  = 0
    query_param_count:   int  = 0
    has_data_uri:        bool = False
    has_javascript_uri:  bool = False
    is_punycode:         bool = False
    path_length:         int  = 0
    valid:               bool = True
    parse_error:         str  = ""


def extract_features(raw_url: str) -> URLFeatures:
    """
    Parse a raw URL string and extract all security-relevant features.
    The URL is NEVER visited or resolved during this process.

    Parameters
    ----------
    raw_url : str
        The URL string to analyse.

    Returns
    -------
    URLFeatures
        Populated dataclass with all extracted features.
    """
    f = URLFeatures(raw_url=raw_url.strip())

    if not f.raw_url:
        f.valid = False
        f.parse_error = "Empty URL"
        return f

    url_lower = f.raw_url.lower()

    # ── Dangerous URI schemes ────────────────────────────────
    if url_lower.startswith("javascript:"):
        f.has_javascript_uri = True
        f.valid = False
        f.parse_error = "javascript: URI — cannot be scanned"
        return f

    if url_lower.startswith("data:"):
        f.has_data_uri = True
        f.valid = False
        f.parse_error = "data: URI — cannot be scanned"
        return f

    # ── Add scheme if missing so urlparse works ──────────────
    working = f.raw_url
    if not re.match(r'^[a-zA-Z][a-zA-Z0-9+\-.]*://', working):
        working = "http://" + working

    try:
        parsed = urlparse(working)
    except Exception as exc:
        f.valid = False
        f.parse_error = str(exc)
        return f

    f.scheme   = parsed.scheme.lower()
    f.hostname = (parsed.hostname or "").lower()
    f.path     = parsed.path
    f.query    = parsed.query
    f.fragment = parsed.fragment

    if not f.hostname:
        f.valid = False
        f.parse_error = "No hostname could be parsed"
        return f

    # ── Basic metrics ────────────────────────────────────────
    f.url_length  = len(f.raw_url)
    f.path_length = len(f.path)
    f.is_https    = f.scheme == "https"

    # ── IP-based hostname ────────────────────────────────────
    f.is_ip_hostname = bool(
        re.match(r'^\d{1,3}\.\d{1,3}\.\d{1,3}\.\d{1,3}$', f.hostname)
    )

    # ── @ symbol in URL (credentials trick) ─────────────────
    f.has_at_symbol = "@" in f.raw_url

    # ── Double slash in path (redirect trick) ───────────────
    f.has_double_slash = "//" in parsed.path

    # ── Subdomain count ──────────────────────────────────────
    parts = f.hostname.split(".")
    # e.g. "a.b.com" → parts=[a,b,com] → 2 registered + 1 sub
    f.subdomain_count = max(0, len(parts) - 2)

    # ── Suspicious keywords ──────────────────────────────────
    full_lower = url_lower
    found_kw = [kw for kw in SUSPICIOUS_KEYWORDS if kw in full_lower]
    f.suspicious_kw_found = found_kw
    f.suspicious_kw_count = len(found_kw)

    # ── Hex / percent encoding ───────────────────────────────
    f.has_hex_encoding = bool(re.search(r'%[0-9a-fA-F]{2}', f.raw_url))

    # ── Port ─────────────────────────────────────────────────
    if parsed.port:
        f.has_port = True
        f.port     = parsed.port

    # ── TLD in path (double domain trick) ───────────────────
    f.has_tld_in_path = bool(
        re.search(r'\.(com|net|org|gov|edu|io)/', f.path.lower())
    )

    # ── Unusual TLD ──────────────────────────────────────────
    if len(parts) >= 2:
        tld_candidate = "." + parts[-1]
        # Check two-level TLDs too
        if len(parts) >= 3:
            two_level = "." + parts[-2] + "." + parts[-1]
            if two_level in COMMON_TLDS:
                f.tld = two_level
            else:
                f.tld = tld_candidate
        else:
            f.tld = tld_candidate
        f.unusual_tld = f.tld not in COMMON_TLDS

    # ── Hostname character metrics ───────────────────────────
    f.dash_count_hostname  = f.hostname.count("-")
    f.dot_count_hostname   = f.hostname.count(".")
    f.digit_count_hostname = sum(c.isdigit() for c in f.hostname)

    # ── Query parameter count ────────────────────────────────
    if f.query:
        f.query_param_count = len(f.query.split("&"))

    # ── Punycode (IDN homograph attack) ─────────────────────
    f.is_punycode = "xn--" in f.hostname

    return f


# Quick smoke-test
if __name__ == "__main__" or True:
    _test = extract_features("http://paypal-secure.login.verify.com/account/update?user=test")
    print("✅ Feature extractor ready.")
    print(f"   Sample hostname : {_test.hostname}")
    print(f"   Keywords found  : {_test.suspicious_kw_found}")
    print(f"   Subdomain count : {_test.subdomain_count}")

In [ ]:
# ============================================================
#  CELL 3 — Risk Classification Engine
# ============================================================

@dataclass
class ScanResult:
    """Complete result for one URL scan."""
    url:           str
    risk_label:    str          # "Low Risk" | "Suspicious" | "High Risk"
    risk_score:    int          # 0–100 (heuristic, NOT a probability)
    reasons:       List[str]
    features:      URLFeatures
    ti_result:     Dict         = field(default_factory=dict)
    scanned_at:    str          = field(default_factory=lambda: datetime.datetime.now().isoformat(timespec="seconds"))
    scan_id:       str          = ""

    def __post_init__(self):
        if not self.scan_id:
            raw = f"{self.url}{self.scanned_at}"
            self.scan_id = hashlib.sha1(raw.encode()).hexdigest()[:12]


def classify_url(features: URLFeatures) -> Tuple[int, str, List[str]]:
    """
    Apply heuristic rules to the extracted features and return
    (risk_score, risk_label, reasons).

    IMPORTANT: The score is a heuristic index (0–100), NOT a
    statistical probability. Do not interpret it as a percentage
    likelihood of being a phishing URL.
    """
    score   = 0
    reasons = []

    if not features.valid:
        return 100, "High Risk", [f"Invalid URL: {features.parse_error}"]

    # ── Rule 1: Protocol ─────────────────────────────────────
    if not features.is_https:
        score += 10
        reasons.append("Uses HTTP (not HTTPS) — connection is not encrypted")

    # ── Rule 2: IP-based hostname ────────────────────────────
    if features.is_ip_hostname:
        score += 30
        reasons.append("Hostname is a raw IP address — legitimate sites use domain names")

    # ── Rule 3: @ symbol ────────────────────────────────────
    if features.has_at_symbol:
        score += 20
        reasons.append("URL contains '@' — browsers ignore everything before '@', used for deception")

    # ── Rule 4: URL length ───────────────────────────────────
    if features.url_length > 100:
        score += 10
        reasons.append(f"URL is very long ({features.url_length} chars) — often used to obscure the real destination")
    elif features.url_length > 75:
        score += 5
        reasons.append(f"URL is moderately long ({features.url_length} chars)")

    # ── Rule 5: Suspicious keywords ─────────────────────────
    kw_count = features.suspicious_kw_count
    if kw_count >= 3:
        score += 20
        reasons.append(f"Contains {kw_count} suspicious keywords: {', '.join(features.suspicious_kw_found[:5])}")
    elif kw_count == 2:
        score += 12
        reasons.append(f"Contains suspicious keywords: {', '.join(features.suspicious_kw_found)}")
    elif kw_count == 1:
        score += 6
        reasons.append(f"Contains suspicious keyword: '{features.suspicious_kw_found[0]}'")

    # ── Rule 6: Subdomains ───────────────────────────────────
    if features.subdomain_count >= 3:
        score += 15
        reasons.append(f"Excessive subdomain depth ({features.subdomain_count}) — often mimics legitimate domains")
    elif features.subdomain_count == 2:
        score += 8
        reasons.append(f"Multiple subdomains ({features.subdomain_count}) detected")

    # ── Rule 7: Dashes in hostname ───────────────────────────
    if features.dash_count_hostname >= 3:
        score += 10
        reasons.append(f"Many hyphens in hostname ({features.dash_count_hostname}) — common in phishing domains")
    elif features.dash_count_hostname == 2:
        score += 5
        reasons.append(f"Multiple hyphens in hostname ({features.dash_count_hostname})")

    # ── Rule 8: Hex/percent encoding ────────────────────────
    if features.has_hex_encoding:
        score += 8
        reasons.append("URL uses percent-encoding — may be obfuscating the real content")

    # ── Rule 9: Double slash in path ─────────────────────────
    if features.has_double_slash:
        score += 10
        reasons.append("Double slash '//' in path — may indicate an open redirect")

    # ── Rule 10: Port ────────────────────────────────────────
    if features.has_port:
        non_std = features.port not in (80, 443, 8080, 8443)
        if non_std:
            score += 10
            reasons.append(f"Non-standard port {features.port} — legitimate sites rarely use unusual ports")
        else:
            score += 3
            reasons.append(f"Explicit port {features.port} specified in URL")

    # ── Rule 11: TLD in path ─────────────────────────────────
    if features.has_tld_in_path:
        score += 12
        reasons.append("TLD appears inside the URL path — classic double-domain deception trick")

    # ── Rule 12: Unusual TLD ─────────────────────────────────
    if features.unusual_tld:
        score += 8
        reasons.append(f"Unusual TLD '{features.tld}' — legitimate organisations rarely use obscure TLDs")

    # ── Rule 13: Punycode ────────────────────────────────────
    if features.is_punycode:
        score += 15
        reasons.append("Punycode (xn--) in hostname — possible IDN homograph attack")

    # ── Rule 14: Digits in hostname ──────────────────────────
    if features.digit_count_hostname >= 4:
        score += 7
        reasons.append(f"Many digits in hostname ({features.digit_count_hostname}) — unusual for legitimate sites")

    # ── Clamp to [0, 100] ────────────────────────────────────
    score = min(score, 100)

    # ── Determine label ──────────────────────────────────────
    lo, hi = RISK_THRESHOLDS["low"]
    if score <= hi:
        label = "Low Risk"
    elif score <= RISK_THRESHOLDS["suspicious"][1]:
        label = "Suspicious"
    else:
        label = "High Risk"

    if not reasons:
        reasons.append("No specific risk indicators detected by heuristic rules")

    return score, label, reasons


def scan_url(raw_url: str, api_key: str = "") -> ScanResult:
    """
    Full pipeline: extract features → classify → optionally query threat intel.
    The URL is NEVER automatically visited.
    """
    features          = extract_features(raw_url)
    score, label, reasons = classify_url(features)
    ti_result         = {}

    if api_key and REQUESTS_AVAILABLE and features.valid:
        ti_result = query_virustotal(raw_url, api_key)

    return ScanResult(
        url        = raw_url,
        risk_label = label,
        risk_score = score,
        reasons    = reasons,
        features   = features,
        ti_result  = ti_result,
    )


# Smoke-test
if __name__ == "__main__" or True:
    _r = scan_url("https://google.com")
    print("✅ Classifier ready.")
    print(f"   Test URL  : {_r.url}")
    print(f"   Label     : {_r.risk_label} (score {_r.risk_score})")

In [ ]:
# ============================================================
#  CELL 4 — Threat Intelligence (VirusTotal API — optional)
# ============================================================

def query_virustotal(url: str, api_key: str, timeout: int = 10) -> Dict:
    """
    Submit a URL to the VirusTotal v3 API for reputation lookup.

    This function is ONLY called when the user explicitly provides
    an API key. It handles:
      - Missing / empty API key
      - requests library not installed
      - Network timeouts
      - HTTP error codes (401 unauthorized, 429 rate-limited, 5xx)
      - Malformed responses

    Parameters
    ----------
    url     : URL to submit (NOT visited, just analysed by VT)
    api_key : VirusTotal v3 API key
    timeout : request timeout in seconds

    Returns
    -------
    dict with keys: status, malicious, suspicious, harmless,
                    undetected, message
    """
    result = {
        "status":     "unavailable",
        "malicious":  0,
        "suspicious": 0,
        "harmless":   0,
        "undetected": 0,
        "message":    "",
    }

    if not api_key or api_key.strip() == "":
        result["message"] = "No API key provided — threat intelligence lookup skipped"
        return result

    if not REQUESTS_AVAILABLE:
        result["message"] = "'requests' library not installed — install with: pip install requests"
        return result

    headers = {"x-apikey": api_key}

    # Step 1: Submit URL for analysis
    try:
        submit_resp = requests.post(
            "https://www.virustotal.com/api/v3/urls",
            headers=headers,
            data={"url": url},
            timeout=timeout,
        )
    except requests.exceptions.Timeout:
        result["message"] = "VirusTotal request timed out — check your internet connection"
        return result
    except requests.exceptions.ConnectionError:
        result["message"] = "Cannot reach VirusTotal — no internet connection or service unavailable"
        return result
    except Exception as exc:
        result["message"] = f"Unexpected error during VirusTotal submission: {exc}"
        return result

    if submit_resp.status_code == 401:
        result["message"] = "VirusTotal API key is invalid or expired (HTTP 401)"
        return result
    if submit_resp.status_code == 429:
        result["message"] = "VirusTotal rate limit reached — wait before retrying (HTTP 429)"
        return result
    if submit_resp.status_code >= 500:
        result["message"] = f"VirusTotal server error (HTTP {submit_resp.status_code})"
        return result
    if submit_resp.status_code != 200:
        result["message"] = f"VirusTotal returned HTTP {submit_resp.status_code}"
        return result

    # Step 2: Retrieve analysis report
    try:
        analysis_id = submit_resp.json()["data"]["id"]
    except (KeyError, ValueError):
        result["message"] = "Could not parse VirusTotal submission response"
        return result

    try:
        report_resp = requests.get(
            f"https://www.virustotal.com/api/v3/analyses/{analysis_id}",
            headers=headers,
            timeout=timeout,
        )
    except requests.exceptions.Timeout:
        result["message"] = "VirusTotal report request timed out"
        return result
    except Exception as exc:
        result["message"] = f"Error retrieving VirusTotal report: {exc}"
        return result

    if report_resp.status_code != 200:
        result["message"] = f"VirusTotal report fetch failed (HTTP {report_resp.status_code})"
        return result

    try:
        stats = report_resp.json()["data"]["attributes"]["stats"]
        result.update({
            "status":     "success",
            "malicious":  stats.get("malicious", 0),
            "suspicious": stats.get("suspicious", 0),
            "harmless":   stats.get("harmless", 0),
            "undetected": stats.get("undetected", 0),
            "message":    "VirusTotal analysis complete",
        })
    except (KeyError, ValueError):
        result["message"] = "Could not parse VirusTotal analysis report"

    return result


print("✅ Threat intelligence module loaded.")
print("   Set VIRUSTOTAL_API_KEY in Cell 1 to enable live lookups.")

In [ ]:
# ============================================================
#  CELL 5 — SQLite Scan History Manager
# ============================================================

class HistoryManager:
    """
    Persists scan results in a local SQLite database.
    No personal information is stored beyond the URL, score,
    label, reasons, and timestamp.
    """

    def __init__(self, db_path: str = DB_PATH):
        self.db_path = db_path
        self._init_db()

    def _init_db(self):
        with sqlite3.connect(self.db_path) as conn:
            conn.execute("""
                CREATE TABLE IF NOT EXISTS scan_history (
                    scan_id       TEXT PRIMARY KEY,
                    url           TEXT NOT NULL,
                    risk_label    TEXT NOT NULL,
                    risk_score    INTEGER NOT NULL,
                    reasons       TEXT NOT NULL,
                    ti_status     TEXT,
                    ti_malicious  INTEGER DEFAULT 0,
                    scanned_at    TEXT NOT NULL
                )
            """)
            conn.commit()

    def save(self, result: ScanResult):
        """Insert or replace a scan result in the database."""
        with sqlite3.connect(self.db_path) as conn:
            conn.execute("""
                INSERT OR REPLACE INTO scan_history
                (scan_id, url, risk_label, risk_score, reasons,
                 ti_status, ti_malicious, scanned_at)
                VALUES (?, ?, ?, ?, ?, ?, ?, ?)
            """, (
                result.scan_id,
                result.url,
                result.risk_label,
                result.risk_score,
                json.dumps(result.reasons),
                result.ti_result.get("status", ""),
                result.ti_result.get("malicious", 0),
                result.scanned_at,
            ))
            conn.commit()

    def load_all(self) -> List[Dict]:
        """Return all scan history records as a list of dicts."""
        with sqlite3.connect(self.db_path) as conn:
            conn.row_factory = sqlite3.Row
            rows = conn.execute(
                "SELECT * FROM scan_history ORDER BY scanned_at DESC"
            ).fetchall()
        records = []
        for row in rows:
            d = dict(row)
            try:
                d["reasons"] = json.loads(d["reasons"])
            except (json.JSONDecodeError, TypeError):
                d["reasons"] = []
            records.append(d)
        return records

    def count(self) -> int:
        """Return total number of scans stored."""
        with sqlite3.connect(self.db_path) as conn:
            return conn.execute("SELECT COUNT(*) FROM scan_history").fetchone()[0]

    def clear(self):
        """Delete all scan history records."""
        with sqlite3.connect(self.db_path) as conn:
            conn.execute("DELETE FROM scan_history")
            conn.commit()

    def stats(self) -> Dict:
        """Aggregate statistics across all stored scans."""
        records = self.load_all()
        if not records:
            return {"total": 0, "low": 0, "suspicious": 0, "high": 0, "avg_score": 0}
        total = len(records)
        return {
            "total":      total,
            "low":        sum(1 for r in records if r["risk_label"] == "Low Risk"),
            "suspicious": sum(1 for r in records if r["risk_label"] == "Suspicious"),
            "high":       sum(1 for r in records if r["risk_label"] == "High Risk"),
            "avg_score":  round(sum(r["risk_score"] for r in records) / total, 1),
        }


# Initialise the global history manager
history_manager = HistoryManager(DB_PATH)
print(f"✅ History manager ready — database: {DB_PATH}")
print(f"   Existing records: {history_manager.count()}")

In [ ]:
# ============================================================
#  CELL 6 — Export Module (CSV & JSON)
# ============================================================

def export_to_csv(records: List[Dict], filepath: str = "phishguard_export.csv") -> str:
    """
    Export scan history records to a CSV file.

    Parameters
    ----------
    records  : list of dicts from HistoryManager.load_all()
    filepath : output file path

    Returns
    -------
    Absolute path to the created file.
    """
    if not records:
        return "No records to export."

    flat_records = []
    for r in records:
        flat = dict(r)
        if isinstance(flat.get("reasons"), list):
            flat["reasons"] = " | ".join(flat["reasons"])
        flat_records.append(flat)

    fieldnames = ["scan_id", "url", "risk_label", "risk_score",
                  "reasons", "ti_status", "ti_malicious", "scanned_at"]

    with open(filepath, "w", newline="", encoding="utf-8") as fh:
        writer = csv.DictWriter(fh, fieldnames=fieldnames, extrasaction="ignore")
        writer.writeheader()
        writer.writerows(flat_records)

    abs_path = str(Path(filepath).resolve())
    print(f"📄 CSV exported: {abs_path} ({len(flat_records)} records)")
    return abs_path


def export_to_json(records: List[Dict], filepath: str = "phishguard_export.json") -> str:
    """
    Export scan history records to a pretty-printed JSON file.

    Parameters
    ----------
    records  : list of dicts from HistoryManager.load_all()
    filepath : output file path

    Returns
    -------
    Absolute path to the created file.
    """
    if not records:
        return "No records to export."

    export_data = {
        "exported_at": datetime.datetime.now().isoformat(timespec="seconds"),
        "tool": "PhishGuard — Intelligent Phishing Detection System",
        "author": "Tushar Solanki",
        "total_records": len(records),
        "records": records,
    }

    with open(filepath, "w", encoding="utf-8") as fh:
        json.dump(export_data, fh, indent=2, default=str)

    abs_path = str(Path(filepath).resolve())
    print(f"📄 JSON exported: {abs_path} ({len(records)} records)")
    return abs_path


def export_single_result(result: ScanResult, fmt: str = "json") -> str:
    """
    Serialise a single ScanResult to a JSON or CSV string (not file).
    Useful for in-notebook display.
    """
    d = {
        "scan_id":    result.scan_id,
        "url":        result.url,
        "risk_label": result.risk_label,
        "risk_score": result.risk_score,
        "reasons":    result.reasons,
        "scanned_at": result.scanned_at,
        "ti_status":  result.ti_result.get("status", "n/a"),
    }
    if fmt.lower() == "json":
        return json.dumps(d, indent=2)
    # CSV fallback
    lines = ["field,value"]
    for k, v in d.items():
        v_str = " | ".join(v) if isinstance(v, list) else str(v)
        lines.append(f'{k},"{v_str}"')
    return "\n".join(lines)


print("✅ Export module ready.")

In [ ]:
# ============================================================
#  CELL 7 — Offline Demo: Sample URLs & Demonstration Data
# ============================================================

# These URLs are used for demonstration ONLY.
# They are NEVER visited or resolved — only their structure is analysed.
# Phishing-like samples are fabricated examples of phishing patterns,
# NOT real verified phishing pages.

DEMO_URLS = [
    # ── Legitimate / Low Risk ────────────────────────────────
    {
        "url":   "https://www.google.com",
        "label": "Legitimate",
        "note":  "Well-known search engine, HTTPS, simple domain",
    },
    {
        "url":   "https://github.com/openai/openai-python",
        "label": "Legitimate",
        "note":  "Popular code repository, HTTPS, standard path",
    },
    {
        "url":   "https://en.wikipedia.org/wiki/Phishing",
        "label": "Legitimate",
        "note":  "Educational resource on phishing",
    },
    {
        "url":   "https://stackoverflow.com/questions/tagged/python",
        "label": "Legitimate",
        "note":  "Developer community site",
    },
    {
        "url":   "https://www.amazon.com/dp/B09KMPL123",
        "label": "Legitimate",
        "note":  "E-commerce product page, HTTPS",
    },

    # ── Suspicious ───────────────────────────────────────────
    {
        "url":   "http://paypal-account-verify.com/login",
        "label": "Suspicious",
        "note":  "HTTP, contains paypal+verify+login keywords, hyphens",
    },
    {
        "url":   "https://secure-bank-update.net/customer/account",
        "label": "Suspicious",
        "note":  "Suspicious keywords: secure, bank, update, account",
    },
    {
        "url":   "http://apple-support-helpdesk.com/verify-id",
        "label": "Suspicious",
        "note":  "Apple brand keyword, support, helpdesk, verify, HTTP",
    },
    {
        "url":   "https://login-microsoft.com.au/signin",
        "label": "Suspicious",
        "note":  "Brand mimicry with login prefix, signin path",
    },

    # ── High Risk ────────────────────────────────────────────
    {
        "url":   "http://192.168.1.200/paypal/login/verify/account/update",
        "label": "High Risk",
        "note":  "IP-based host, HTTP, multiple phishing keywords",
    },
    {
        "url":   "http://confirm-paypal-account.verify-login.secure-update.xyz/webscr?cmd=signin&user=victim@example.com",
        "label": "High Risk",
        "note":  "Extreme keyword density, @ symbol, unusual TLD, HTTP, very long URL",
    },
    {
        "url":   "http://216.58.200.100:8888/secure/banking/credential/signin",
        "label": "High Risk",
        "note":  "IP host, non-standard port, multiple phishing keywords, HTTP",
    },
    {
        "url":   "https://xn--pple-43d.com/signin/apple-id/verify",
        "label": "High Risk",
        "note":  "Punycode IDN homograph attack (apple → xn--pple), signin, verify",
    },
    {
        "url":   "http://amazon.login-verify.com.secure.account-update.info/confirm",
        "label": "High Risk",
        "note":  "Excessive subdomains, Amazon brand, multiple keywords, unusual TLD",
    },
]


def run_demo(seed_history: bool = True, verbose: bool = True) -> List[ScanResult]:
    """
    Run all DEMO_URLS through the PhishGuard pipeline.
    Optionally seed the scan history with results.
    """
    results = []
    for sample in DEMO_URLS:
        r = scan_url(sample["url"])
        results.append(r)
        if seed_history:
            history_manager.save(r)
        if verbose:
            icon = {"Low Risk": "🟢", "Suspicious": "🟡", "High Risk": "🔴"}.get(r.risk_label, "⚪")
            print(f"{icon} [{r.risk_label:12s}] score={r.risk_score:3d}  {r.url[:70]}")
    return results


print("✅ Demo data module loaded.  Running demo scans...")
print("-" * 70)
demo_results = run_demo(seed_history=True, verbose=True)
print("-" * 70)
print(f"\nDemo scans complete. History now contains {history_manager.count()} records.")

In [ ]:
# ============================================================
#  CELL 8 — Result Display Helpers
# ============================================================

def _risk_color(label: str) -> str:
    return {"Low Risk": "#27ae60", "Suspicious": "#e67e22", "High Risk": "#c0392b"}.get(label, "#7f8c8d")


def print_result(result: ScanResult):
    """Plain-text result display (works without ipywidgets)."""
    color_map = {"Low Risk": "🟢", "Suspicious": "🟡", "High Risk": "🔴"}
    icon = color_map.get(result.risk_label, "⚪")
    print("\n" + "═" * 65)
    print(f"  PhishGuard Scan Result  —  {result.scanned_at}")
    print("═" * 65)
    print(f"  URL        : {result.url}")
    print(f"  Verdict    : {icon} {result.risk_label}")
    print(f"  Risk Score : {result.risk_score}/100  (heuristic index, NOT a probability)")
    print(f"  Scan ID    : {result.scan_id}")
    print()
    print("  Risk Indicators:")
    for i, reason in enumerate(result.reasons, 1):
        print(f"    {i:2d}. {reason}")
    print()
    f = result.features
    print("  Feature Summary:")
    print(f"    Hostname       : {f.hostname or 'N/A'}")
    print(f"    Scheme         : {f.scheme or 'N/A'}")
    print(f"    HTTPS          : {'Yes' if f.is_https else 'No'}")
    print(f"    IP Hostname    : {'Yes' if f.is_ip_hostname else 'No'}")
    print(f"    URL Length     : {f.url_length}")
    print(f"    Subdomains     : {f.subdomain_count}")
    print(f"    Suspicious KW  : {f.suspicious_kw_count}  ({', '.join(f.suspicious_kw_found) or 'none'})")
    print(f"    Port           : {f.port or 'standard'}")
    if result.ti_result.get("status") == "success":
        ti = result.ti_result
        print()
        print("  Threat Intelligence (VirusTotal):")
        print(f"    Malicious  : {ti['malicious']}")
        print(f"    Suspicious : {ti['suspicious']}")
        print(f"    Harmless   : {ti['harmless']}")
        print(f"    Undetected : {ti['undetected']}")
    elif result.ti_result.get("message"):
        print(f"\n  Threat Intelligence : {result.ti_result['message']}")
    print("═" * 65)


def display_html_result(result: ScanResult):
    """Rich HTML result card (used inside widgets / notebook output)."""
    if not WIDGETS_AVAILABLE:
        print_result(result)
        return

    color     = _risk_color(result.risk_label)
    icon      = {"Low Risk": "🟢", "Suspicious": "🟡", "High Risk": "🔴"}.get(result.risk_label, "⚪")
    f         = result.features
    reasons_html = "".join(f"<li>{r}</li>" for r in result.reasons)
    kw_str    = ", ".join(f.suspicious_kw_found) or "none"
    ti_block  = ""
    if result.ti_result.get("status") == "success":
        ti = result.ti_result
        ti_block = f"""
        <div style="margin-top:12px;border-top:1px solid #ddd;padding-top:10px">
          <strong>Threat Intelligence (VirusTotal)</strong>
          <table style="font-size:13px;margin-top:6px">
            <tr><td>Malicious:</td><td style="color:#c0392b;font-weight:bold">{ti['malicious']}</td></tr>
            <tr><td>Suspicious:</td><td style="color:#e67e22">{ti['suspicious']}</td></tr>
            <tr><td>Harmless:</td><td style="color:#27ae60">{ti['harmless']}</td></tr>
            <tr><td>Undetected:</td><td style="color:#7f8c8d">{ti['undetected']}</td></tr>
          </table>
        </div>"""
    elif result.ti_result.get("message"):
        ti_block = f"<div style='margin-top:10px;font-size:12px;color:#7f8c8d'>Threat Intel: {result.ti_result['message']}</div>"

    bar_width = result.risk_score
    bar_color = color

    html = f"""
    <div style="font-family:Arial,sans-serif;max-width:720px;border:2px solid {color};
                border-radius:10px;padding:20px;margin:10px 0;background:#fafafa">
      <div style="display:flex;align-items:center;justify-content:space-between">
        <div>
          <span style="font-size:22px;font-weight:bold;color:{color}">{icon} {result.risk_label}</span>
          <span style="margin-left:16px;font-size:13px;color:#7f8c8d">Score: {result.risk_score}/100 (heuristic)</span>
        </div>
        <div style="font-size:11px;color:#aaa">{result.scanned_at}</div>
      </div>

      <div style="margin:8px 0 4px;background:#e0e0e0;border-radius:6px;height:10px">
        <div style="width:{bar_width}%;height:10px;background:{bar_color};border-radius:6px"></div>
      </div>
      <div style="font-size:11px;color:#aaa;text-align:right">{bar_width}% heuristic risk index</div>

      <div style="margin-top:10px;word-break:break-all;font-size:13px">
        <strong>URL:</strong> <code>{result.url}</code>
      </div>

      <div style="margin-top:12px">
        <strong>Risk Indicators:</strong>
        <ul style="margin:4px 0 0 16px;font-size:13px">{reasons_html}</ul>
      </div>

      <div style="margin-top:12px;display:grid;grid-template-columns:1fr 1fr;gap:4px;font-size:13px">
        <div><strong>Hostname:</strong> {f.hostname or 'N/A'}</div>
        <div><strong>Scheme:</strong> {f.scheme or 'N/A'}</div>
        <div><strong>HTTPS:</strong> {'✅ Yes' if f.is_https else '❌ No'}</div>
        <div><strong>IP Hostname:</strong> {'⚠️ Yes' if f.is_ip_hostname else 'No'}</div>
        <div><strong>URL Length:</strong> {f.url_length} chars</div>
        <div><strong>Subdomains:</strong> {f.subdomain_count}</div>
        <div><strong>Keywords ({f.suspicious_kw_count}):</strong> {kw_str}</div>
        <div><strong>Port:</strong> {f.port or 'standard'}</div>
      </div>
      {ti_block}
      <div style="margin-top:10px;font-size:11px;color:#aaa;border-top:1px solid #ddd;padding-top:8px">
        ⚠️ Heuristic scores are NOT probabilities and can produce false positives/negatives.
        Scan ID: {result.scan_id}
      </div>
    </div>
    """
    display(HTML(html))


print("✅ Display helpers ready.")

In [ ]:
# ============================================================
#  CELL 9 — Dashboard Charts
# ============================================================

def plot_risk_distribution(records: List[Dict], title: str = "Risk Distribution"):
    """Pie chart of Low/Suspicious/High Risk counts."""
    if not MATPLOTLIB_AVAILABLE:
        print("[Charts disabled — matplotlib not installed]")
        return
    low  = sum(1 for r in records if r["risk_label"] == "Low Risk")
    susp = sum(1 for r in records if r["risk_label"] == "Suspicious")
    high = sum(1 for r in records if r["risk_label"] == "High Risk")
    labels  = []
    sizes   = []
    colors  = []
    for lbl, cnt, col in [("Low Risk", low, "#27ae60"),
                          ("Suspicious", susp, "#e67e22"),
                          ("High Risk", high, "#c0392b")]:
        if cnt > 0:
            labels.append(f"{lbl} ({cnt})")
            sizes.append(cnt)
            colors.append(col)
    if not sizes:
        print("No data to plot.")
        return
    fig, ax = plt.subplots(figsize=(6, 4))
    wedges, texts, autotexts = ax.pie(
        sizes, labels=labels, colors=colors,
        autopct="%1.0f%%", startangle=90,
        textprops={"fontsize": 11},
    )
    ax.set_title(title, fontsize=13, fontweight="bold", pad=14)
    plt.tight_layout()
    plt.show()


def plot_score_histogram(records: List[Dict], title: str = "Risk Score Distribution"):
    """Histogram of heuristic risk scores."""
    if not MATPLOTLIB_AVAILABLE:
        return
    scores = [r["risk_score"] for r in records]
    if not scores:
        print("No scores to plot.")
        return
    fig, ax = plt.subplots(figsize=(7, 4))
    bins = list(range(0, 110, 10))
    n, bins_out, patches = ax.hist(scores, bins=bins, color="#3498db", edgecolor="white", rwidth=0.85)
    # Colour patches by risk zone
    for patch, left in zip(patches, bins_out):
        if left < 40:
            patch.set_facecolor("#27ae60")
        elif left < 70:
            patch.set_facecolor("#e67e22")
        else:
            patch.set_facecolor("#c0392b")
    ax.set_xlabel("Heuristic Risk Score (0–100)", fontsize=11)
    ax.set_ylabel("Number of URLs", fontsize=11)
    ax.set_title(title, fontsize=13, fontweight="bold")
    ax.set_xticks(range(0, 110, 10))
    # Legend
    legend = [
        mpatches.Patch(color="#27ae60", label="Low Risk (0–39)"),
        mpatches.Patch(color="#e67e22", label="Suspicious (40–69)"),
        mpatches.Patch(color="#c0392b", label="High Risk (70+)"),
    ]
    ax.legend(handles=legend, fontsize=9)
    plt.tight_layout()
    plt.show()


def plot_feature_frequency(results: List[ScanResult], title: str = "Feature Frequency"):
    """Bar chart of how often each risk feature was triggered."""
    if not MATPLOTLIB_AVAILABLE:
        return
    counts = {
        "No HTTPS":       0,
        "IP Hostname":    0,
        "@ Symbol":       0,
        "Long URL": 0,
        "Susp. Keywords": 0,
        "3+ Subdomains":  0,
        "Hex Encoding":   0,
        "Double Slash":   0,
        "Non-std Port":   0,
        "TLD in Path":    0,
        "Unusual TLD":    0,
        "Punycode":       0,
    }
    for r in results:
        f = r.features
        if not f.valid:
            continue
        if not f.is_https:                   counts["No HTTPS"]       += 1
        if f.is_ip_hostname:                 counts["IP Hostname"]    += 1
        if f.has_at_symbol:                  counts["@ Symbol"]       += 1
        if f.url_length > 75:                counts["Long URL"] += 1
        if f.suspicious_kw_count > 0:        counts["Susp. Keywords"] += 1
        if f.subdomain_count >= 3:           counts["3+ Subdomains"]  += 1
        if f.has_hex_encoding:               counts["Hex Encoding"]   += 1
        if f.has_double_slash:               counts["Double Slash"]   += 1
        if f.has_port and f.port not in (80, 443, 8080, 8443, None):  counts["Non-std Port"] += 1
        if f.has_tld_in_path:                counts["TLD in Path"]    += 1
        if f.unusual_tld:                    counts["Unusual TLD"]    += 1
        if f.is_punycode:                    counts["Punycode"]       += 1

    labels = list(counts.keys())
    values = list(counts.values())

    fig, ax = plt.subplots(figsize=(9, 4))
    bars = ax.barh(labels, values, color="#e74c3c", edgecolor="white")
    ax.set_xlabel("Number of URLs triggering this feature", fontsize=10)
    ax.set_title(title, fontsize=13, fontweight="bold")
    ax.bar_label(bars, padding=3, fontsize=9)
    plt.tight_layout()
    plt.show()


def show_summary_stats(records: List[Dict]):
    """Print a text summary of scan statistics."""
    s = history_manager.stats()
    print("\n" + "─" * 45)
    print("  PhishGuard — Scan Summary Statistics")
    print("─" * 45)
    print(f"  Total scans   : {s['total']}")
    print(f"  Low Risk      : {s['low']:4d}  ({100*s['low']//max(s['total'],1):3d}%)")
    print(f"  Suspicious    : {s['suspicious']:4d}  ({100*s['suspicious']//max(s['total'],1):3d}%)")
    print(f"  High Risk     : {s['high']:4d}  ({100*s['high']//max(s['total'],1):3d}%)")
    print(f"  Avg Score     : {s['avg_score']}")
    print("─" * 45 + "\n")


# Render the demo charts immediately
if MATPLOTLIB_AVAILABLE:
    import matplotlib
    matplotlib.rcParams['figure.dpi'] = 110
    all_records = history_manager.load_all()
    show_summary_stats(all_records)
    plot_risk_distribution(all_records, "Demo: Risk Distribution")
    plot_score_histogram(all_records,   "Demo: Heuristic Score Distribution")
    plot_feature_frequency(demo_results, "Demo: Feature Frequency Across Sample URLs")
else:
    print("[Charts skipped — install matplotlib with: pip install matplotlib]")
    show_summary_stats(history_manager.load_all())

print("✅ Dashboard charts ready.")

In [ ]:
# ============================================================
#  CELL 10 — Interactive URL Scanner
# ============================================================

def interactive_scanner():
    """
    Launch the interactive PhishGuard URL scanner.
    Requires ipywidgets + a running Jupyter kernel.
    Falls back to a simple input() prompt if unavailable.
    """

    # ── Fallback: plain input() ──────────────────────────────
    if not WIDGETS_AVAILABLE:
        print("Interactive widgets not available. Using plain input mode.")
        print("Type 'quit' to exit.\n")
        while True:
            try:
                raw = input("Enter URL to scan: ").strip()
            except (EOFError, KeyboardInterrupt):
                break
            if raw.lower() in ("quit", "exit", "q", ""):
                break
            result = scan_url(raw, api_key=VIRUSTOTAL_API_KEY)
            history_manager.save(result)
            print_result(result)
        return

    # ── Widget-based UI ──────────────────────────────────────
    title_html = widgets.HTML(
        value="""
        <div style="font-family:Arial,sans-serif;background:linear-gradient(135deg,#1a1a2e,#16213e);
                    color:white;padding:20px 24px;border-radius:10px;margin-bottom:10px">
          <h2 style="margin:0;font-size:22px">🛡️ PhishGuard — URL Scanner</h2>
          <p style="margin:6px 0 0;font-size:13px;color:#aaa">
            Enter any URL to analyse its risk heuristically. The URL is <strong>never visited</strong>.
          </p>
        </div>"""
    )

    url_input = widgets.Text(
        placeholder="https://example.com/path?param=value",
        description="URL:",
        layout=widgets.Layout(width="600px"),
        style={"description_width": "50px"},
    )

    scan_btn   = widgets.Button(description="🔍 Scan",       button_style="primary",  layout=widgets.Layout(width="120px"))
    clear_btn  = widgets.Button(description="🗑️ Clear Output", button_style="warning", layout=widgets.Layout(width="150px"))
    export_csv_btn  = widgets.Button(description="📥 Export CSV",  button_style="success", layout=widgets.Layout(width="140px"))
    export_json_btn = widgets.Button(description="📥 Export JSON", button_style="success", layout=widgets.Layout(width="140px"))
    clear_hist_btn  = widgets.Button(description="🗑️ Clear History", button_style="danger", layout=widgets.Layout(width="150px"))

    output_area = widgets.Output()

    def on_scan(b):
        raw = url_input.value.strip()
        if not raw:
            with output_area:
                display(HTML("<p style='color:orange'>⚠️ Please enter a URL.</p>"))
            return
        with output_area:
            display(HTML("<p style='color:#aaa;font-size:12px'>Scanning…</p>"))
        result = scan_url(raw, api_key=VIRUSTOTAL_API_KEY)
        history_manager.save(result)
        with output_area:
            clear_output(wait=True)
            display_html_result(result)

    def on_clear_output(b):
        with output_area:
            clear_output()

    def on_export_csv(b):
        records = history_manager.load_all()
        path = export_to_csv(records)
        with output_area:
            display(HTML(f"<p style='color:green'>✅ CSV saved: <code>{path}</code></p>"))

    def on_export_json(b):
        records = history_manager.load_all()
        path = export_to_json(records)
        with output_area:
            display(HTML(f"<p style='color:green'>✅ JSON saved: <code>{path}</code></p>"))

    def on_clear_hist(b):
        history_manager.clear()
        with output_area:
            clear_output()
            display(HTML("<p style='color:#c0392b'>🗑️ Scan history cleared.</p>"))

    url_input.on_submit(on_scan)          # Enter key triggers scan
    scan_btn.on_click(on_scan)
    clear_btn.on_click(on_clear_output)
    export_csv_btn.on_click(on_export_csv)
    export_json_btn.on_click(on_export_json)
    clear_hist_btn.on_click(on_clear_hist)

    row1 = widgets.HBox([url_input, scan_btn, clear_btn])
    row2 = widgets.HBox([export_csv_btn, export_json_btn, clear_hist_btn])

    display(title_html, row1, row2, output_area)


# Launch the scanner
interactive_scanner()

In [ ]:
# ============================================================
#  CELL 11 — Scan History Viewer
# ============================================================

def show_history(limit: int = 20):
    """Display the most recent scan history records."""
    records = history_manager.load_all()[:limit]
    if not records:
        print("No scan history available.")
        return

    if WIDGETS_AVAILABLE:
        icon_map = {"Low Risk": "🟢", "Suspicious": "🟡", "High Risk": "🔴"}
        rows_html = ""
        for r in records:
            icon  = icon_map.get(r["risk_label"], "⚪")
            color = _risk_color(r["risk_label"])
            url_short = r["url"][:65] + ("…" if len(r["url"]) > 65 else "")
            rows_html += f"""
            <tr>
              <td style="padding:6px 8px;border-bottom:1px solid #eee;font-size:12px;color:#555">{r['scanned_at']}</td>
              <td style="padding:6px 8px;border-bottom:1px solid #eee;font-size:12px;word-break:break-all">
                <code>{url_short}</code></td>
              <td style="padding:6px 8px;border-bottom:1px solid #eee;font-weight:bold;color:{color}">
                {icon} {r['risk_label']}</td>
              <td style="padding:6px 8px;border-bottom:1px solid #eee;text-align:center">{r['risk_score']}</td>
            </tr>"""

        html = f"""
        <div style="font-family:Arial,sans-serif;margin:10px 0">
          <h3 style="margin-bottom:8px">📋 Scan History (last {min(limit, len(records))} records)</h3>
          <table style="border-collapse:collapse;width:100%;font-size:13px">
            <thead>
              <tr style="background:#f0f0f0">
                <th style="padding:8px;text-align:left">Timestamp</th>
                <th style="padding:8px;text-align:left">URL</th>
                <th style="padding:8px;text-align:left">Verdict</th>
                <th style="padding:8px;text-align:center">Score</th>
              </tr>
            </thead>
            <tbody>{rows_html}</tbody>
          </table>
        </div>"""
        display(HTML(html))
    else:
        icon_map = {"Low Risk": "🟢", "Suspicious": "🟡", "High Risk": "🔴"}
        print(f"\n{'Timestamp':<22} {'Verdict':<14} {'Score':>5}  URL")
        print("─" * 90)
        for r in records:
            icon  = icon_map.get(r["risk_label"], "⚪")
            url_short = r["url"][:50] + ("…" if len(r["url"]) > 50 else "")
            print(f"{r['scanned_at']:<22} {icon} {r['risk_label']:<12} {r['risk_score']:>5}  {url_short}")


show_history(limit=20)
print("\n✅ History viewer ready.")

In [ ]:
# ============================================================
#  CELL 12 — Manual Export Demo
# ============================================================

print("Exporting scan history...")
all_records = history_manager.load_all()

csv_path  = export_to_csv(all_records,  filepath="phishguard_export.csv")
json_path = export_to_json(all_records, filepath="phishguard_export.json")

print("\nSingle result JSON export sample:")
print("-" * 45)
if demo_results:
    print(export_single_result(demo_results[0], fmt="json"))

print("\n✅ Export demo complete.")

In [ ]:
# ============================================================
#  CELL 13 — Automated Tests
# ============================================================

class TestURLFeatureExtractor(unittest.TestCase):
    """Tests for the feature extraction engine."""

    def test_https_detection(self):
        f = extract_features("https://example.com")
        self.assertTrue(f.is_https, "HTTPS scheme should be detected as True")

    def test_http_not_https(self):
        f = extract_features("http://example.com")
        self.assertFalse(f.is_https, "HTTP should not be flagged as HTTPS")

    def test_ip_hostname(self):
        f = extract_features("http://192.168.0.1/admin")
        self.assertTrue(f.is_ip_hostname)

    def test_domain_not_ip(self):
        f = extract_features("https://google.com")
        self.assertFalse(f.is_ip_hostname)

    def test_at_symbol(self):
        f = extract_features("http://attacker.com@victim.com/path")
        self.assertTrue(f.has_at_symbol)

    def test_no_at_symbol(self):
        f = extract_features("https://normal.com/page")
        self.assertFalse(f.has_at_symbol)

    def test_subdomain_count(self):
        f = extract_features("https://a.b.c.evil.com")
        self.assertGreaterEqual(f.subdomain_count, 2)

    def test_suspicious_keywords(self):
        f = extract_features("https://login.verify.paypal.com.evil.xyz/signin")
        self.assertGreater(f.suspicious_kw_count, 0)
        self.assertIn("login", f.suspicious_kw_found)

    def test_hex_encoding(self):
        f = extract_features("http://example.com/path%20with%20spaces")
        self.assertTrue(f.has_hex_encoding)

    def test_punycode(self):
        f = extract_features("https://xn--pple-43d.com")
        self.assertTrue(f.is_punycode)

    def test_unusual_tld(self):
        f = extract_features("http://example.xyz")
        self.assertTrue(f.unusual_tld)

    def test_common_tld_not_unusual(self):
        f = extract_features("https://example.com")
        self.assertFalse(f.unusual_tld)

    def test_port_extraction(self):
        f = extract_features("http://example.com:8080/path")
        self.assertTrue(f.has_port)
        self.assertEqual(f.port, 8080)

    def test_url_length(self):
        url = "http://example.com/" + "a" * 90
        f = extract_features(url)
        self.assertGreater(f.url_length, 100)

    def test_empty_url(self):
        f = extract_features("")
        self.assertFalse(f.valid)
        self.assertIn("Empty", f.parse_error)

    def test_javascript_uri(self):
        f = extract_features("javascript:alert(1)")
        self.assertFalse(f.valid)
        self.assertTrue(f.has_javascript_uri)

    def test_data_uri(self):
        f = extract_features("data:text/html,<script>alert(1)</script>")
        self.assertFalse(f.valid)
        self.assertTrue(f.has_data_uri)

    def test_scheme_added_if_missing(self):
        f = extract_features("google.com")
        self.assertTrue(f.valid, "Should still parse if scheme is missing")
        self.assertIn("google", f.hostname)


class TestRiskClassifier(unittest.TestCase):
    """Tests for the risk classification engine."""

    def test_google_low_risk(self):
        r = scan_url("https://www.google.com")
        self.assertIn(r.risk_label, ("Low Risk", "Suspicious"),
                      "google.com should not be High Risk")
        self.assertLess(r.risk_score, 70)

    def test_ip_url_high_score(self):
        r = scan_url("http://10.0.0.1/paypal/login")
        self.assertGreaterEqual(r.risk_score, 40,
                                "IP+HTTP+keywords should score at least 40")

    def test_invalid_url_high_risk(self):
        r = scan_url("")
        self.assertEqual(r.risk_label, "High Risk")
        self.assertEqual(r.risk_score, 100)

    def test_score_in_range(self):
        for url in ["https://google.com", "http://192.168.1.1", "", "javascript:x"]:
            r = scan_url(url)
            self.assertGreaterEqual(r.risk_score, 0)
            self.assertLessEqual(r.risk_score, 100, f"Score out of range for {url!r}")

    def test_reasons_not_empty(self):
        for url in ["https://google.com", "http://1.2.3.4/login"]:
            r = scan_url(url)
            self.assertGreater(len(r.reasons), 0, f"Reasons empty for {url!r}")

    def test_label_matches_score(self):
        r = scan_url("https://example.com")
        if r.risk_score <= 39:
            self.assertEqual(r.risk_label, "Low Risk")
        elif r.risk_score <= 69:
            self.assertEqual(r.risk_label, "Suspicious")
        else:
            self.assertEqual(r.risk_label, "High Risk")

    def test_scan_id_generated(self):
        r = scan_url("https://example.com")
        self.assertTrue(len(r.scan_id) > 0)

    def test_keyword_density_increases_score(self):
        r_none = scan_url("https://example.com")
        r_many = scan_url("https://paypal-login-verify-account-secure.com/signin/confirm")
        self.assertGreater(r_many.risk_score, r_none.risk_score)


class TestHistoryManager(unittest.TestCase):
    """Tests for the SQLite scan history."""

    def setUp(self):
        self.test_db = ":memory:"
        self.hm = HistoryManager(self.test_db)

    def test_save_and_load(self):
        r = scan_url("https://test.com")
        self.hm.save(r)
        records = self.hm.load_all()
        self.assertEqual(len(records), 1)
        self.assertEqual(records[0]["url"], "https://test.com")

    def test_clear(self):
        self.hm.save(scan_url("https://test.com"))
        self.hm.clear()
        self.assertEqual(self.hm.count(), 0)

    def test_count(self):
        for i in range(3):
            self.hm.save(scan_url(f"https://test{i}.com"))
        self.assertEqual(self.hm.count(), 3)

    def test_stats_empty(self):
        s = self.hm.stats()
        self.assertEqual(s["total"], 0)

    def test_stats_populated(self):
        self.hm.save(scan_url("https://google.com"))
        self.hm.save(scan_url("http://192.168.0.1/login"))
        s = self.hm.stats()
        self.assertEqual(s["total"], 2)


class TestExport(unittest.TestCase):
    """Tests for CSV/JSON export."""

    def setUp(self):
        self.hm = HistoryManager(":memory:")
        self.hm.save(scan_url("https://example.com"))
        self.hm.save(scan_url("http://192.168.1.1/login"))

    def test_export_csv_creates_file(self):
        path = "_test_export.csv"
        export_to_csv(self.hm.load_all(), filepath=path)
        self.assertTrue(os.path.exists(path))
        os.remove(path)

    def test_export_json_creates_file(self):
        path = "_test_export.json"
        export_to_json(self.hm.load_all(), filepath=path)
        self.assertTrue(os.path.exists(path))
        with open(path, encoding="utf-8") as fh:
            data = json.load(fh)
        self.assertIn("records", data)
        self.assertEqual(data["total_records"], 2)
        os.remove(path)

    def test_export_empty_records(self):
        result = export_to_csv([], filepath="_test_empty.csv")
        self.assertIn("No records", result)

    def test_single_result_json(self):
        r = scan_url("https://example.com")
        out = export_single_result(r, fmt="json")
        data = json.loads(out)
        self.assertIn("url", data)
        self.assertIn("risk_label", data)


class TestEdgeCases(unittest.TestCase):
    """Edge cases and adversarial inputs."""

    def test_very_long_url(self):
        long_url = "http://example.com/" + "x" * 500
        r = scan_url(long_url)
        self.assertGreaterEqual(r.risk_score, 0)

    def test_url_with_only_spaces(self):
        r = scan_url("   ")
        self.assertEqual(r.risk_label, "High Risk")

    def test_unicode_url(self):
        f = extract_features("https://üñíçödé.example.com")
        self.assertIsInstance(f.url_length, int)

    def test_no_path(self):
        f = extract_features("https://example.com")
        self.assertTrue(f.valid)

    def test_deeply_nested_path(self):
        f = extract_features("https://example.com/a/b/c/d/e/f/g")
        self.assertTrue(f.valid)

    def test_port_443_not_flagged_high(self):
        r = scan_url("https://example.com:443/page")
        self.assertLess(r.risk_score, 70, "Port 443 on HTTPS should not trigger high risk")

    def test_ftp_scheme(self):
        f = extract_features("ftp://files.example.com/data.zip")
        self.assertIsInstance(f, URLFeatures)


# ── Run all tests ────────────────────────────────────────────
print("\n" + "═" * 60)
print("  PhishGuard — Automated Test Suite")
print("═" * 60)

loader = unittest.TestLoader()
suite  = unittest.TestSuite()
for tc in [TestURLFeatureExtractor, TestRiskClassifier,
           TestHistoryManager, TestExport, TestEdgeCases]:
    suite.addTests(loader.loadTestsFromTestCase(tc))

runner = unittest.TextTestRunner(verbosity=2, stream=sys.stdout)
result_obj = runner.run(suite)

total  = result_obj.testsRun
failed = len(result_obj.failures) + len(result_obj.errors)
passed = total - failed

print("\n" + "─" * 60)
print(f"  Results: {passed}/{total} tests passed", end="")
if failed == 0:
    print("  ✅  ALL TESTS PASSED")
else:
    print(f"  ❌  {failed} FAILED")
print("─" * 60)

## 📚 13. Usage Guide

### Running PhishGuard

1. **Run cells in order** — each cell builds on the previous one.
2. **Scan a URL** — use the interactive scanner in Cell 10, or call `scan_url()` directly:
   ```python
   result = scan_url("https://suspicious-site.example.com/login")
   print_result(result)
   ```
3. **View history** — call `show_history()` or `history_manager.load_all()`.
4. **Export data** — call `export_to_csv(records)` or `export_to_json(records)`.
5. **Threat intelligence** — set `VIRUSTOTAL_API_KEY` in Cell 1 to enable.
6. **Run tests** — re-run Cell 13 at any time.

### Understanding the Risk Score

| Score Range | Label      | Meaning |
|---|---|---|
| 0 – 39     | 🟢 Low Risk   | Few or no heuristic indicators detected |
| 40 – 69    | 🟡 Suspicious | Some risk indicators; exercise caution |
| 70 – 100   | 🔴 High Risk  | Multiple strong indicators; do not proceed |

> **Important:** The score is a **heuristic index**, not a mathematical probability.  
> A score of 75 does **not** mean the URL is 75% likely to be phishing.  
> Always apply human judgment.

### API Key Setup (Optional)

```python
# In Cell 1, set:
VIRUSTOTAL_API_KEY = "your_key_here"
```

Get a free API key at: https://www.virustotal.com/gui/join-us

---

## 🔬 14. Sample Demonstrations

```python
# Quick single scan
result = scan_url("http://paypal-secure-account.verify.com/login")
print_result(result)

# Batch scan a list of URLs
urls = [
    "https://github.com",
    "http://192.168.0.1/admin/login",
    "https://amazon.signin-verify.info/account/confirm",
]
for url in urls:
    r = scan_url(url)
    history_manager.save(r)
    print(f"{r.risk_label:12s} score={r.risk_score:3d}  {url}")

# Show updated history & charts
all_records = history_manager.load_all()
show_summary_stats(all_records)
plot_risk_distribution(all_records)

# Export everything
export_to_csv(all_records)
export_to_json(all_records)
```

---

## 📝 15. Conclusion

PhishGuard demonstrates that a meaningful level of phishing URL analysis can be performed **entirely offline and without ever visiting the target URL**, using only structural and lexical heuristics.

**Key takeaways:**
- Heuristics are transparent and explainable but imperfect.
- Real-world systems combine multiple signals: heuristics + ML + threat feeds.
- User education remains the most effective phishing defence.
- Privacy is preserved: no credentials or sensitive data are collected.

**Author:** Tushar Solanki  
**Project:** PhishGuard — Intelligent Phishing Detection System  